## Notebook Architecture:
00. Project configuration
01. Install / import libraries
02. Dataset A — SAMSum
03. Preprocessing + tokenization
04. Experiment 1A — Transformer from scratch
05. Experiment 1B — torch.nn.Transformer
06. Experiment 2 — pretrained BART on SAMSum
07. Experiment 3 — BART CNN/DM → SAMSum
08. Evaluation
09. Qualitative examples
10. Results + discussion

## 0

In [1]:
!pip install -q datasets transformers evaluate sentencepiece transformers[torch] rouge_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00


## 1

In [2]:
import math
import random
import time
import os
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

import evaluate

Configuration

In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

Device: cuda


Data Loading

In [4]:
samsum = load_dataset("knkarthick/samsum")
print(samsum["train"][0])
print(samsum)

README.md: 0.00B [00:00, ?B/s]

train.csv: 0.00B [00:00, ?B/s]

validation.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/14731 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/818 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/819 [00:00<?, ? examples/s]

{'id': '13818513', 'dialogue': "Amanda: I baked  cookies. Do you want some?\nJerry: Sure!\nAmanda: I'll bring you tomorrow :-)", 'summary': 'Amanda baked cookies and will bring Jerry some tomorrow.'}
DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
})


Dataset inspection

In [5]:
for i in range(2):
    print("=" * 80)
    print("DIALOGUE:")
    print(samsum["train"][i]["dialogue"])
    print("\nSUMMARY:")
    print(samsum["train"][i]["summary"])

DIALOGUE:
Amanda: I baked  cookies. Do you want some?
Jerry: Sure!
Amanda: I'll bring you tomorrow :-)

SUMMARY:
Amanda baked cookies and will bring Jerry some tomorrow.
DIALOGUE:
Olivia: Who are you voting for in this election? 
Oliver: Liberals as always.
Olivia: Me too!!
Oliver: Great

SUMMARY:
Olivia and Olivier are voting for liberals in this election. 


Smaller experimental subset

In [6]:
TRAIN_SIZE = 10000
VAL_SIZE = 1000
TEST_SIZE = 1000

train_raw = samsum["train"].select(range(min(TRAIN_SIZE, len(samsum["train"]))))
val_raw = samsum["validation"].select(range(min(VAL_SIZE, len(samsum["validation"]))))
test_raw = samsum["test"].select(range(min(TEST_SIZE, len(samsum["test"]))))

print(len(train_raw), len(val_raw), len(test_raw))

10000 818 819


Tokenizer for 1A and 1B

In [7]:
from collections import Counter

SPECIAL_TOKENS = [
    "<pad>",
    "<unk>",
    "<bos>",
    "<eos>"
]

PAD_IDX = 0
UNK_IDX = 1
BOS_IDX = 2
EOS_IDX = 3

def tokenize(text):
    return text.lower().strip().split()

counter = Counter()

for example in train_raw:
    counter.update(tokenize(example["dialogue"]))
    counter.update(tokenize(example["summary"]))

MAX_VOCAB_SIZE = 20000

most_common = counter.most_common(MAX_VOCAB_SIZE - len(SPECIAL_TOKENS))

itos = SPECIAL_TOKENS + [word for word, freq in most_common]

stoi = {word: idx for idx, word in enumerate(itos)}

VOCAB_SIZE = len(itos)

print("Vocabulary size:", VOCAB_SIZE)

# encoding
def encode_text(text):
    tokens = tokenize(text)

    ids = [
        stoi.get(token, UNK_IDX)
        for token in tokens
    ]

    return [BOS_IDX] + ids + [EOS_IDX]

# decoding
def decode_ids(ids):
    words = []

    for idx in ids:
        if idx in [PAD_IDX, BOS_IDX]:
            continue

        if idx == EOS_IDX:
            break

        words.append(itos[idx])

    return " ".join(words)

# quick test
text = train_raw[0]["summary"]

encoded = encode_text(text)
print()
print(text)
print(encoded)
print(decode_ids(encoded))

Vocabulary size: 20000

Amanda baked cookies and will bring Jerry some tomorrow.
[2, 1931, 5569, 2630, 9, 15, 182, 2334, 46, 278, 3]
amanda baked cookies and will bring jerry some tomorrow.


Dataset class

In [8]:
class SummarizationDataset(Dataset):
    def __init__(self, hf_dataset, max_src_len=128, max_tgt_len=64):
        self.dataset = hf_dataset
        self.max_src_len = max_src_len
        self.max_tgt_len = max_tgt_len

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]

        src = encode_text(item["dialogue"])
        tgt = encode_text(item["summary"])

        src = src[:self.max_src_len]
        tgt = tgt[:self.max_tgt_len]

        return {
            "src": torch.tensor(src, dtype=torch.long),
            "tgt": torch.tensor(tgt, dtype=torch.long)
        }

padding collate function

In [9]:
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):

    src_batch = [item["src"] for item in batch]
    tgt_batch = [item["tgt"] for item in batch]

    src_batch = pad_sequence(
        src_batch,
        padding_value=PAD_IDX,
        batch_first=False
    )

    tgt_batch = pad_sequence(
        tgt_batch,
        padding_value=PAD_IDX,
        batch_first=False
    )

    return src_batch, tgt_batch


BATCH_SIZE = 64

train_dataset = SummarizationDataset(train_raw)
val_dataset = SummarizationDataset(val_raw)
test_dataset = SummarizationDataset(test_raw)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

## Exp. 1A: Model implementation - from scratch

Positional encoding
$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{\frac{2i}{d_{\text{model}}}}}\right)$$

$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{\frac{2i}{d_{\text{model}}}}}\right)$$


In [10]:
class PositionalEncoding(nn.Module):

    def __init__(self, d_model, dropout=0.1, max_len=5000):

        super().__init__()

        self.dropout = nn.Dropout(dropout)

        position = torch.arange(max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2)
            * (-math.log(10000.0) / d_model)
        )

        pe = torch.zeros(max_len, d_model)

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(1)

        self.register_buffer("pe", pe)

    def forward(self, x):

        x = x + self.pe[:x.size(0)]

        return self.dropout(x)

Multi-Head-Attention
$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \text{head}_2, \dots, \text{head}_h)W^O$$
where
$$\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V) = \text{softmax}\left(\frac{(QW_i^Q)(KW_i^K)^T}{\sqrt{d_k}}\right)(VW_i^V)$$

In [11]:
class MultiHeadAttention(nn.Module):

    def __init__(self, d_model, num_heads, dropout=0.1):

        super().__init__()

        assert d_model % num_heads == 0, f"d_model ({d_model}) must be dividable by num_heads ({num_heads})"

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)

        self.out_proj = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        query,
        key,
        value,
        mask=None
    ):

        batch_size = query.size(1)

        # query/key/value:
        # [seq_len, batch, d_model]

        Q = self.q_proj(query)
        K = self.k_proj(key)
        V = self.v_proj(value)

        # [seq_len, batch, heads, head_dim]
        Q = Q.view(
            -1,
            batch_size,
            self.num_heads,
            self.head_dim
        )

        K = K.view(
            -1,
            batch_size,
            self.num_heads,
            self.head_dim
        )

        V = V.view(
            -1,
            batch_size,
            self.num_heads,
            self.head_dim
        )

        # [batch, heads, seq_len, head_dim]
        Q = Q.permute(1, 2, 0, 3)
        K = K.permute(1, 2, 0, 3)
        V = V.permute(1, 2, 0, 3)

        # Attention scores
        scores = torch.matmul(
            Q,
            K.transpose(-2, -1)
        )

        scores = scores / math.sqrt(self.head_dim)

        if mask is not None:
            scores = scores.masked_fill(
                mask,
                float("-inf")
            )

        attention = torch.softmax(scores, dim=-1)

        attention = self.dropout(attention)

        output = torch.matmul(
            attention,
            V
        )

        # [batch, heads, seq_len, head_dim]
        output = output.permute(2, 0, 1, 3)

        # [seq_len, batch, d_model]
        output = output.contiguous().view(
            -1,
            batch_size,
            self.d_model
        )

        return self.out_proj(output)

FeedFoward network

In [12]:
class FeedForward(nn.Module):

    def __init__(self, d_model, d_ff, dropout=0.1):

        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )

    def forward(self, x):
        return self.net(x)

Encoder

In [13]:
class EncoderLayer(nn.Module):

    def __init__(
        self,
        d_model,
        num_heads,
        d_ff,
        dropout=0.1
    ):

        super().__init__()

        self.self_attn = MultiHeadAttention(
            d_model,
            num_heads,
            dropout
        )

        self.ffn = FeedForward(
            d_model,
            d_ff,
            dropout
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_mask=None):

        attn = self.self_attn(
            x,
            x,
            x,
            src_mask
        )

        x = self.norm1(
            x + self.dropout(attn)
        )

        ffn = self.ffn(x)

        x = self.norm2(
            x + self.dropout(ffn)
        )

        return x

Decoder

In [14]:
class DecoderLayer(nn.Module):

    def __init__(
        self,
        d_model,
        num_heads,
        d_ff,
        dropout=0.1
    ):

        super().__init__()

        self.self_attn = MultiHeadAttention(
            d_model,
            num_heads,
            dropout
        )

        self.cross_attn = MultiHeadAttention(
            d_model,
            num_heads,
            dropout
        )

        self.ffn = FeedForward(
            d_model,
            d_ff,
            dropout
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        x,
        memory,
        tgt_mask=None,
        memory_mask=None
    ):

        attn = self.self_attn(
            x,
            x,
            x,
            tgt_mask
        )

        x = self.norm1(
            x + self.dropout(attn)
        )

        attn = self.cross_attn(
            x,
            memory,
            memory,
            memory_mask
        )

        x = self.norm2(
            x + self.dropout(attn)
        )

        ffn = self.ffn(x)

        x = self.norm3(
            x + self.dropout(ffn)
        )

        return x

Full Transformer

In [15]:
class TransformerFromScratch(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model=256,
        num_heads=8,
        num_layers=4,
        d_ff=1024,
        dropout=0.1,
        max_len=5000
    ):

        super().__init__()

        self.d_model = d_model

        self.src_embedding = nn.Embedding(
            vocab_size,
            d_model
        )

        self.tgt_embedding = nn.Embedding(
            vocab_size,
            d_model
        )

        self.positional_encoding = PositionalEncoding(
            d_model,
            dropout,
            max_len
        )

        self.encoder_layers = nn.ModuleList([
            EncoderLayer(
                d_model,
                num_heads,
                d_ff,
                dropout
            )
            for _ in range(num_layers)
        ])

        self.decoder_layers = nn.ModuleList([
            DecoderLayer(
                d_model,
                num_heads,
                d_ff,
                dropout
            )
            for _ in range(num_layers)
        ])

        self.output_projection = nn.Linear(
            d_model,
            vocab_size
        )

    def encode(self, src, src_mask=None):

        x = self.src_embedding(src)
        x = x * math.sqrt(self.d_model)

        x = self.positional_encoding(x)

        for layer in self.encoder_layers:
            x = layer(x, src_mask)

        return x

    def decode(
        self,
        tgt,
        memory,
        tgt_mask=None
    ):

        x = self.tgt_embedding(tgt)
        x = x * math.sqrt(self.d_model)

        x = self.positional_encoding(x)

        for layer in self.decoder_layers:

            x = layer(
                x,
                memory,
                tgt_mask
            )

        return x

    def forward(
        self,
        src,
        tgt,
        src_mask=None,
        tgt_mask=None
    ):

        memory = self.encode(
            src,
            src_mask
        )

        output = self.decode(
            tgt,
            memory,
            tgt_mask
        )

        return self.output_projection(output)

Causal mask and Padding mask

In [16]:
def generate_square_subsequent_mask(size, device):

    mask = torch.triu(
        torch.ones(
            size,
            size,
            device=device
        ),
        diagonal=1
    ).bool()

    return mask

def create_padding_mask(x):
    return (x == PAD_IDX)

Instantiate Model

In [17]:
scratch_model = TransformerFromScratch(
    vocab_size=VOCAB_SIZE,
    d_model=256,
    num_heads=8,
    num_layers=4,
    d_ff=1024,
    dropout=0.1
).to(device)

print(
    f"Parameters: "
    f"{sum(p.numel() for p in scratch_model.parameters()):,}"
)

Parameters: 22,752,800


Loss function & Optimizer

In [18]:
criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_IDX
)

optimizer = optim.Adam(
    scratch_model.parameters(),
    lr=3e-4,
    betas=(0.9, 0.98),
    eps=1e-9
)

Training Loop & Validation

In [19]:
def train_one_epoch(
	model,
	loader,
	optimizer,
	criterion
):

	model.train()
	print("Training started!")
	total_loss = 0

	for src, tgt in loader:

		src = src.to(device)
		tgt = tgt.to(device)

		optimizer.zero_grad()

		tgt_input = tgt[:-1]
		tgt_output = tgt[1:]

		tgt_mask = generate_square_subsequent_mask(
			tgt_input.size(0),
			device
		)

		logits = model(
			src,
			tgt_input,
			tgt_mask=tgt_mask
		)

		loss = criterion(
			logits.reshape(-1, VOCAB_SIZE),
			tgt_output.reshape(-1)
		)

		loss.backward()

		torch.nn.utils.clip_grad_norm_(
			model.parameters(),
			1.0
		)

		optimizer.step()

		total_loss += loss.item()

	return total_loss / len(loader)

@torch.no_grad()
def evaluate_loss(
	model,
	loader,
	criterion
):

	model.eval()

	total_loss = 0

	for src, tgt in loader:

		src = src.to(device)
		tgt = tgt.to(device)

		tgt_input = tgt[:-1]
		tgt_output = tgt[1:]

		tgt_mask = generate_square_subsequent_mask(
			tgt_input.size(0),
			device
		)

		logits = model(
			src,
			tgt_input,
			tgt_mask=tgt_mask
		)

		loss = criterion(
			logits.reshape(-1, VOCAB_SIZE),
			tgt_output.reshape(-1)
		)

		total_loss += loss.item()

	return total_loss / len(loader)

Train

In [20]:
EPOCHS = 20 # Modify this

scratch_history = []

import os
import time

# Create directory for checkpoints
os.makedirs("checkpoints", exist_ok=True)

best_val_loss = float("inf")

for epoch in range(EPOCHS):

    start = time.time()

    train_loss = train_one_epoch(
        scratch_model,
        train_loader,
        optimizer,
        criterion
    )

    val_loss = evaluate_loss(
        scratch_model,
        val_loader,
        criterion
    )

    elapsed = time.time() - start

    scratch_history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss
    })

    # -------------------------
    # Save LAST model
    # -------------------------
    last_checkpoint = {
        "epoch": epoch + 1,
        "model_state_dict": scratch_model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_loss": train_loss,
        "val_loss": val_loss,
        "history": scratch_history,
    }

    torch.save(
        last_checkpoint,
        "checkpoints/last_model.pt"
    )

    # -------------------------
    # Save BEST model
    # -------------------------
    if val_loss < best_val_loss:

        best_val_loss = val_loss

        best_checkpoint = {
            "epoch": epoch + 1,
            "model_state_dict": scratch_model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "train_loss": train_loss,
            "val_loss": val_loss,
            "history": scratch_history,
        }

        torch.save(
            best_checkpoint,
            "checkpoints/best_model.pt"
        )

        print("  → New best model saved!")

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Train: {train_loss:.4f} | "
        f"Val: {val_loss:.4f} | "
        f"Time: {elapsed:.1f}s"
    )

Training started!
  → New best model saved!
Epoch 1/20 | Train: 6.7564 | Val: 5.7802 | Time: 26.6s
Training started!
  → New best model saved!
Epoch 2/20 | Train: 5.7141 | Val: 5.3723 | Time: 26.4s
Training started!
  → New best model saved!
Epoch 3/20 | Train: 5.3381 | Val: 5.1624 | Time: 27.4s
Training started!
  → New best model saved!
Epoch 4/20 | Train: 5.0979 | Val: 5.0546 | Time: 28.6s
Training started!
  → New best model saved!
Epoch 5/20 | Train: 4.9225 | Val: 4.9805 | Time: 29.3s
Training started!
  → New best model saved!
Epoch 6/20 | Train: 4.7645 | Val: 4.9117 | Time: 28.3s
Training started!
  → New best model saved!
Epoch 7/20 | Train: 4.6163 | Val: 4.8665 | Time: 28.6s
Training started!
  → New best model saved!
Epoch 8/20 | Train: 4.4652 | Val: 4.8051 | Time: 28.8s
Training started!
  → New best model saved!
Epoch 9/20 | Train: 4.3016 | Val: 4.7463 | Time: 28.5s
Training started!
  → New best model saved!
Epoch 10/20 | Train: 4.1391 | Val: 4.7013 | Time: 28.6s
Training 

Greedy decoding & Test

In [21]:
@torch.no_grad()
def summarize_scratch(
    model,
    text,
    max_len=64
):

    model.eval()

    src_ids = encode_text(text)

    src = torch.tensor(
        src_ids,
        dtype=torch.long,
        device=device
    ).unsqueeze(1)

    memory = model.encode(src)

    generated = torch.tensor(
        [[BOS_IDX]],
        dtype=torch.long,
        device=device
    )

    for _ in range(max_len):

        tgt_mask = generate_square_subsequent_mask(
            generated.size(0),
            device
        )

        decoder_output = model.decode(
            generated,
            memory,
            tgt_mask
        )

        logits = model.output_projection(
            decoder_output[-1]
        )

        next_token = logits.argmax(
            dim=-1
        ).item()

        generated = torch.cat([
            generated,
            torch.tensor(
                [[next_token]],
                device=device
            )
        ])

        if next_token == EOS_IDX:
            break

    return decode_ids(
        generated.squeeze(1).tolist()
    )

example = test_raw[0]

print("DIALOGUE:")
print(example["dialogue"])

print("\nREFERENCE:")
print(example["summary"])

print("\nMODEL:")
print(summarize_scratch(
    scratch_model,
    example["dialogue"]
))

DIALOGUE:
Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye

REFERENCE:
Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.

MODEL:
hannah is looking for a place where hannah is not sure if she hasn't seen her phone number from her last night. hannah is going to the last night. hannah will ask hannah and hannah will ask hannah tonight.


As you can see, the model is speaking Garbage

## Exp 1B: Model pre-implementated - from scratch

In [22]:
class TorchTransformer(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model=256,
        num_heads=8,
        num_layers=4,
        d_ff=1024,
        dropout=0.1
    ):

        super().__init__()

        self.d_model = d_model

        self.src_embedding = nn.Embedding(
            vocab_size,
            d_model
        )

        self.tgt_embedding = nn.Embedding(
            vocab_size,
            d_model
        )

        self.positional_encoding = PositionalEncoding(
            d_model,
            dropout
        )

        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=num_heads,
            num_encoder_layers=num_layers,
            num_decoder_layers=num_layers,
            dim_feedforward=d_ff,
            dropout=dropout,
            batch_first=False
        )

        self.output_projection = nn.Linear(
            d_model,
            vocab_size
        )

    def forward(
        self,
        src,
        tgt,
        tgt_mask=None
    ):

        src = self.src_embedding(src)
        tgt = self.tgt_embedding(tgt)

        src = src * math.sqrt(self.d_model)
        tgt = tgt * math.sqrt(self.d_model)

        src = self.positional_encoding(src)
        tgt = self.positional_encoding(tgt)

        output = self.transformer(
            src,
            tgt,
            tgt_mask=tgt_mask
        )

        return self.output_projection(output)

Model Instantiate

In [23]:
torch_transformer = TorchTransformer(
    vocab_size=VOCAB_SIZE,
    d_model=256,
    num_heads=8,
    num_layers=4,
    d_ff=1024,
    dropout=0.1
).to(device)

torch_optimizer = optim.Adam(
    torch_transformer.parameters(),
    lr=3e-4
)

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder = TransformerEncoder(


Re-used training function

In [24]:
torch_history = []

import os

# Create directory for checkpoints
os.makedirs("checkpoints/torch_transformer", exist_ok=True)

best_val_loss = float("inf")

for epoch in range(EPOCHS):

    train_loss = train_one_epoch(
        torch_transformer,
        train_loader,
        torch_optimizer,
        criterion
    )

    val_loss = evaluate_loss(
        torch_transformer,
        val_loader,
        criterion
    )

    torch_history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss
    })

    # -------------------------
    # Save LAST model
    # -------------------------
    last_checkpoint = {
        "epoch": epoch + 1,
        "model_state_dict": torch_transformer.state_dict(),
        "optimizer_state_dict": torch_optimizer.state_dict(),
        "train_loss": train_loss,
        "val_loss": val_loss,
        "history": torch_history,
    }

    torch.save(
        last_checkpoint,
        "checkpoints/torch_transformer/last_model.pt"
    )

    # -------------------------
    # Save BEST model
    # -------------------------
    if val_loss < best_val_loss:

        best_val_loss = val_loss

        best_checkpoint = {
            "epoch": epoch + 1,
            "model_state_dict": torch_transformer.state_dict(),
            "optimizer_state_dict": torch_optimizer.state_dict(),
            "train_loss": train_loss,
            "val_loss": val_loss,
            "history": torch_history,
        }

        torch.save(
            best_checkpoint,
            "checkpoints/torch_transformer/best_model.pt"
        )

        print("  → New best model saved!")

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Train: {train_loss:.4f} | "
        f"Val: {val_loss:.4f}"
    )

Training started!
  → New best model saved!
Epoch 1/20 | Train: 6.8840 | Val: 5.9244
Training started!
  → New best model saved!
Epoch 2/20 | Train: 5.9158 | Val: 5.5299
Training started!
  → New best model saved!
Epoch 3/20 | Train: 5.5616 | Val: 5.3128
Training started!
  → New best model saved!
Epoch 4/20 | Train: 5.3304 | Val: 5.1770
Training started!
  → New best model saved!
Epoch 5/20 | Train: 5.1489 | Val: 5.0729
Training started!
  → New best model saved!
Epoch 6/20 | Train: 5.0077 | Val: 5.0213
Training started!
  → New best model saved!
Epoch 7/20 | Train: 4.8796 | Val: 4.9527
Training started!
  → New best model saved!
Epoch 8/20 | Train: 4.7635 | Val: 4.9023
Training started!
  → New best model saved!
Epoch 9/20 | Train: 4.6524 | Val: 4.8682
Training started!
  → New best model saved!
Epoch 10/20 | Train: 4.5443 | Val: 4.8346
Training started!
  → New best model saved!
Epoch 11/20 | Train: 4.4366 | Val: 4.8127
Training started!
  → New best model saved!
Epoch 12/20 | Train

Test model

In [25]:
@torch.no_grad()
def summarize_torch_transformer(
    model,
    text,
    max_len=64
):

    model.eval()

    # -------------------------
    # Encode source text
    # -------------------------
    src_ids = encode_text(text)

    src = torch.tensor(
        src_ids,
        dtype=torch.long,
        device=device
    ).unsqueeze(1)   # [src_len, 1]

    # -------------------------
    # Start with BOS
    # -------------------------
    generated = torch.tensor(
        [[BOS_IDX]],
        dtype=torch.long,
        device=device
    )   # [1, 1]

    # -------------------------
    # Autoregressive generation
    # -------------------------
    for _ in range(max_len):

        tgt_mask = generate_square_subsequent_mask(
            generated.size(0),
            device
        )

        # Forward pass
        output = model(
            src,
            generated,
            tgt_mask=tgt_mask
        )
        # output shape:
        # [tgt_len, batch_size, vocab_size]

        # Get logits for the newest token
        logits = output[-1, 0]
        # [vocab_size]

        # Greedy decoding
        next_token = logits.argmax(
            dim=-1
        ).item()

        # Append predicted token
        generated = torch.cat(
            [
                generated,
                torch.tensor(
                    [[next_token]],
                    dtype=torch.long,
                    device=device
                )
            ],
            dim=0
        )

        # Stop at EOS
        if next_token == EOS_IDX:
            break

    # -------------------------
    # Decode token IDs
    # -------------------------
    return decode_ids(
        generated.squeeze(1).tolist()
    )

In [26]:
example = test_raw[0]

print("DIALOGUE:")
print(example["dialogue"])

print("\nREFERENCE:")
print(example["summary"])

print("\nMODEL:")
print(summarize_torch_transformer(
    torch_transformer,
    example["dialogue"]
))

DIALOGUE:
Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye

REFERENCE:
Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.

MODEL:
<unk> is not feeling well as he was supposed to tell him about his <unk> he is not feeling well and he is not feeling well.


Still pretty shitty

## Exp. 2: Pretrained model - BART

In [27]:
MODEL_NAME = "facebook/bart-base"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

bart = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
).to(device)

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

Prepare SAMsum

In [28]:
MAX_INPUT_LENGTH = 256
MAX_TARGET_LENGTH = 64

def preprocess_bart(examples):

    inputs = examples["dialogue"]
    targets = examples["summary"]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True
    )

    labels = tokenizer(
        text_target=targets,
        max_length=MAX_TARGET_LENGTH,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

bart_train = train_raw.map(
    preprocess_bart,
    batched=True,
    remove_columns=train_raw.column_names
)

bart_val = val_raw.map(
    preprocess_bart,
    batched=True,
    remove_columns=val_raw.column_names
)

bart_test = test_raw.map(
    preprocess_bart,
    batched=True,
    remove_columns=test_raw.column_names
)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

Map:   0%|          | 0/819 [00:00<?, ? examples/s]

Data Collator

In [29]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=bart
)

BART Fine-tuning

In [30]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./bart-samsum",
    eval_strategy="epoch",
    save_strategy="epoch",

    save_total_limit=1,     # keep only one checkpoint
    save_only_model=True,   # omit optimizer state; much smaller files

    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,

    num_train_epochs=5,     # start with 3–5, not 20
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    report_to="none",
)

# Trainer
trainer = Seq2SeqTrainer(
    model=bart,
    args=training_args,

    train_dataset=bart_train,
    eval_dataset=bart_val,

    processing_class=tokenizer,

    data_collator=data_collator
)
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,3.766476,3.252468
2,3.507104,3.175415
3,3.251711,3.129941
4,3.025923,3.128259
5,2.972515,3.128738


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3125, training_loss=3.3579509619140624, metrics={'train_runtime': 1425.5858, 'train_samples_per_second': 35.073, 'train_steps_per_second': 2.192, 'total_flos': 7553853492756480.0, 'train_loss': 3.3579509619140624, 'epoch': 5.0})

In [31]:
@torch.no_grad()
def summarize_bart(text):

    bart.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        max_length=MAX_INPUT_LENGTH,
        truncation=True
    ).to(device)

    output = bart.generate(
        **inputs,
        max_length=MAX_TARGET_LENGTH,
        num_beams=4,
        early_stopping=True
    )

    return tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

Test 

In [32]:
example = test_raw[0]

print("DIALOGUE:")
print(example["dialogue"])

print("\nREFERENCE:")
print(example["summary"])

print("\nBART:")
print(summarize_bart(example["dialogue"]))

DIALOGUE:
Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye

REFERENCE:
Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.

BART:
Betty called Larry last time they were at the park together.


## Exp. 3: Transfer Learning - BART --> CNN/DailyMail

In [33]:
bart_cnn = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
).to(device)

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

In [34]:
cnn_dm = load_dataset(
    "abisee/cnn_dailymail",
    "3.0.0"
)

cnn_dm

README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

3.0.0/validation-00000-of-00001.parquet:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 11490
    })
})

Smaller Subset

In [35]:
CNN_TRAIN_SIZE = 10000
CNN_VAL_SIZE = 1000

cnn_train = cnn_dm["train"].select(
    range(min(CNN_TRAIN_SIZE, len(cnn_dm["train"])))
)

cnn_val = cnn_dm["validation"].select(
    range(min(CNN_VAL_SIZE, len(cnn_dm["validation"])))
)

CNN/DailyMail Preprocessing

In [36]:
def preprocess_cnn(examples):

    inputs = examples["article"]
    targets = examples["highlights"]

    model_inputs = tokenizer(
        inputs,
        max_length=512,
        truncation=True
    )

    labels = tokenizer(
        text_target=targets,
        max_length=128,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [37]:
cnn_train_tokenized = cnn_train.map(
    preprocess_cnn,
    batched=True,
    remove_columns=cnn_train.column_names
)

cnn_val_tokenized = cnn_val.map(
    preprocess_cnn,
    batched=True,
    remove_columns=cnn_val.column_names
)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Finetune BART one CNN/DailyMail

In [38]:
cnn_train_tokenized = cnn_train.map(
    preprocess_cnn,
    batched=True,
    remove_columns=cnn_train.column_names
)

cnn_val_tokenized = cnn_val.map(
    preprocess_cnn,
    batched=True,
    remove_columns=cnn_val.column_names
)

In [39]:
cnn_args = Seq2SeqTrainingArguments(
    output_dir="./bart-cnn",
    eval_strategy="epoch",
    save_strategy="epoch",

    save_total_limit=1,     # keep only one checkpoint
    save_only_model=True,   # omit optimizer state; much smaller files

    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,

    num_train_epochs=3,     # start with 3–5, not 20
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    report_to="none",
)

In [40]:
cnn_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=bart_cnn
)

cnn_trainer = Seq2SeqTrainer(
    model=bart_cnn,

    args=cnn_args,

    train_dataset=cnn_train_tokenized,

    eval_dataset=cnn_val_tokenized,

    processing_class=tokenizer,

    data_collator=cnn_collator
)
cnn_trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,4.550102,4.378224
2,4.164229,4.339430
3,4.088664,4.330435


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1875, training_loss=4.38218154296875, metrics={'train_runtime': 1490.9493, 'train_samples_per_second': 20.121, 'train_steps_per_second': 1.258, 'total_flos': 9146046873600000.0, 'train_loss': 4.38218154296875, 'epoch': 3.0})

Transfer CNN-trained BART -> SAMSum

In [41]:
transfer_args = Seq2SeqTrainingArguments(
    output_dir="./bart-cnn-samsum",
    eval_strategy="epoch",
    save_strategy="epoch",

    save_total_limit=1,     # keep only one checkpoint
    save_only_model=True,   # omit optimizer state; much smaller files

    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,

    num_train_epochs=5,     # start with 3–5, not 20
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    report_to="none",
)

In [42]:
transfer_trainer = Seq2SeqTrainer(
    model=bart_cnn,

    args=transfer_args,

    train_dataset=bart_train,

    eval_dataset=bart_val,

    processing_class=tokenizer,

    data_collator=data_collator
)

transfer_trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,3.727392,3.248920
2,3.467885,3.164664
3,3.208428,3.124335
4,2.990084,3.124830
5,2.938329,3.123974


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3125, training_loss=3.30840119140625, metrics={'train_runtime': 1426.1836, 'train_samples_per_second': 35.059, 'train_steps_per_second': 2.191, 'total_flos': 7553853492756480.0, 'train_loss': 3.30840119140625, 'epoch': 5.0})

## Evaluation

In [43]:
rouge = evaluate.load("rouge")

# Helper
def evaluate_model(
    model,
    tokenizer,
    dataset,
    batch_size=8
):

    predictions = []
    references = []

    model.eval()

    for i in range(0, len(dataset), batch_size):

        batch = dataset[
            i:min(i + batch_size, len(dataset))
        ]

        inputs = tokenizer(
            batch["dialogue"],
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_INPUT_LENGTH
        ).to(device)

        with torch.no_grad():

            generated = model.generate(
                **inputs,
                max_length=MAX_TARGET_LENGTH,
                num_beams=4
            )

        decoded_preds = tokenizer.batch_decode(
            generated,
            skip_special_tokens=True
        )

        predictions.extend(decoded_preds)
        references.extend(batch["summary"])

    scores = rouge.compute(
        predictions=predictions,
        references=references,
        use_stemmer=True
    )

    return scores, predictions, references

BART

In [44]:
bart_scores, bart_preds, references = evaluate_model(
    bart,
    tokenizer,
    test_raw
)

bart_scores

{'rouge1': np.float64(0.48375648810937233),
 'rouge2': np.float64(0.24037598330882687),
 'rougeL': np.float64(0.40262714282822765),
 'rougeLsum': np.float64(0.4023286013414622)}

Transferred BART

In [45]:
transfer_scores, transfer_preds, references = evaluate_model(
    bart_cnn,
    tokenizer,
    test_raw
)

transfer_scores

{'rouge1': np.float64(0.4880330167991376),
 'rouge2': np.float64(0.24459613770300465),
 'rougeL': np.float64(0.40743346482381254),
 'rougeLsum': np.float64(0.407315130587819)}

Scratch Transformer

In [46]:
scratch_predictions = []
scratch_references = []

for example in test_raw:

    prediction = summarize_scratch(
        scratch_model,
        example["dialogue"]
    )

    scratch_predictions.append(prediction)
    scratch_references.append(example["summary"])

In [47]:
scratch_scores = rouge.compute(
    predictions=scratch_predictions,
    references=scratch_references,
    use_stemmer=True
)

scratch_scores

{'rouge1': np.float64(0.2085218540171273),
 'rouge2': np.float64(0.043712613847742804),
 'rougeL': np.float64(0.1626123515838523),
 'rougeLsum': np.float64(0.16257471401246443)}

Pre-implemented

In [48]:
torch_predictions = []
torch_references = []

for example in test_raw:

    prediction = summarize_torch_transformer(
        torch_transformer,
        example["dialogue"]
    )

    torch_predictions.append(prediction)
    torch_references.append(example["summary"])

In [49]:
torch_scores = rouge.compute(
    predictions=scratch_predictions,
    references=scratch_references,
    use_stemmer=True
)
torch_scores

{'rouge1': np.float64(0.2085218540171273),
 'rouge2': np.float64(0.043712613847742804),
 'rougeL': np.float64(0.1626123515838523),
 'rougeLsum': np.float64(0.16257471401246443)}

In [50]:
results = pd.DataFrame([
    {
        "Model": "Transformer - Scratch",
        "ROUGE-1": scratch_scores["rouge1"],
        "ROUGE-2": scratch_scores["rouge2"],
        "ROUGE-L": scratch_scores["rougeL"]
    },

    {
        "Model": "torch.nn.Transformer",
        "ROUGE-1": torch_scores["rouge1"],
        "ROUGE-2": torch_scores["rouge2"],
        "ROUGE-L": torch_scores["rougeL"]
    },

    {
        "Model": "BART → SAMSum",
        "ROUGE-1": bart_scores["rouge1"],
        "ROUGE-2": bart_scores["rouge2"],
        "ROUGE-L": bart_scores["rougeL"]
    },

    {
        "Model": "BART → CNN/DM → SAMSum",
        "ROUGE-1": transfer_scores["rouge1"],
        "ROUGE-2": transfer_scores["rouge2"],
        "ROUGE-L": transfer_scores["rougeL"]
    }
])

results

,Model,ROUGE-1,ROUGE-2,ROUGE-L
0,Transformer - Scratch,0.208522,0.043713,0.162612
1,torch.nn.Transformer,0.208522,0.043713,0.162612
2,BART → SAMSum,0.483756,0.240376,0.402627
3,BART → CNN/DM → SAMSum,0.488033,0.244596,0.407433


## Acknowledgement
Thanks [Bogdan et. al](https://arxiv.org/abs/1911.12237v2) for the SAMSum dataset, [facebook](https://huggingface.co/facebook/bart-base) for the pretrained BART base model and [abisee](https://huggingface.co/datasets/abisee/cnn_dailymail) for CNN/DailyMail dataset 